In [6]:
from hetero_isas.monodromy_lp.invariants import recover_local_equivalence
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from qiskit.synthesis.two_qubit import TwoQubitWeylDecomposition
from hetero_isas.zz_parallel_drive.bgate import BGate

In [7]:
def b_sandwich(target, local_equiv=False):
    # https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.93.020502
    target_decomp = TwoQubitWeylDecomposition(target, fidelity=1.0)
    c1, c2, c3 = target_decomp.a, target_decomp.b, target_decomp.c
    rb1 = 1 - 4 * np.sin(c2) ** 2 * np.cos(c3) ** 2
    b1 = np.arccos(rb1)
    rb2 = np.sqrt(
        np.cos(2 * c2) * np.cos(2 * c3) / (1 - 2 * np.sin(c2) ** 2 * np.cos(c3) ** 2)
    )
    b2 = np.arcsin(rb2)
    temp = QuantumCircuit(2)
    temp.append(BGate(), [0, 1])
    temp.ry(-2 * c1, 0)
    temp.rz(-b2, 1)
    temp.ry(-b1, 1)
    temp.rz(-b2, 1)
    temp.append(BGate(), [0, 1])

    if local_equiv:
        return recover_local_equivalence(target, Operator(temp))
    return temp